In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import json
GEOSPHERE_RENAME_MAP = {}
from enum import Enum
class _DateCol(Enum):
    DATE = 'date'
class Columns:
    DATE = _DateCol.DATE
def _extract_parameter_values(param_dict):
    for name, info in param_dict.items():
        for v in (info.get('data') or []):
            yield {'parameter': name, 'value': v}
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- geosphere_all ---
GEOSPHERE_RENAME_MAP = {"id": "station_id", "Stationsname": "name"}
_GEOSPHERE_TEST_DATA_DIR = next(
    candidate
    for root in [Path.cwd(), *Path.cwd().parents]
    for candidate in (
        root / "data/generated_outputs/earthobservations__wetterdienst/test/3c49cad1/test_data",
        root / "generated_outputs/earthobservations__wetterdienst/test/3c49cad1/test_data",
    )
    if (candidate / "geosphere_stations.csv").exists()
)
_GEOSPHERE_STATIONS_CSV = _GEOSPHERE_TEST_DATA_DIR / "geosphere_stations.csv"
_GEOSPHERE_VALUES_JSON = _GEOSPHERE_TEST_DATA_DIR / "geosphere_values.json"

def FIX_GEOSPHERE_ALL_RESPONSE():
    return _GEOSPHERE_STATIONS_CSV.open("r", encoding="utf-8", newline="")

# --- geosphere_collect ---
FIX_GEOSPHERE_COLLECT_PARAMETER = ["temperature_air_mean_2m"]
def _extract_parameter_values(parameters):
    rows = []
    for parameter, payload in parameters.items():
        rows.append({"parameter": parameter, "value": payload.get("data", [])})
    return rows
Columns = SimpleNamespace(DATE=SimpleNamespace(value="date"))
class _LocalJsonResponse:
    def __init__(self, path):
        self.path = Path(path)
    def read(self):
        return self.path.read_bytes()
FIX_GEOSPHERE_COLLECT_RESPONSE = _LocalJsonResponse(_GEOSPHERE_VALUES_JSON)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_geosphere_all(response):
    df = pd.read_csv(response)
    return df.rename(
        columns=GEOSPHERE_RENAME_MAP
    ).drop(columns=["Sonnenschein", "Globalstrahlung"])
    return df

def before_geosphere_collect(parameter, response):
    data = json.loads(response.read())
    timestamps = data.pop("timestamps")
    df = (
        pd.DataFrame(data["features"])
        .pop("properties")
        .map(lambda x: x["parameters"])
        .apply(_extract_parameter_values)
        .explode()
        .apply(pd.Series)
        .explode("value")
    )
    df.value = df.value.astype(float)
    df[Columns.DATE.value] = pd.to_datetime(pd.Series(timestamps).repeat(len(parameter)).values)
    return df
    return df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_geosphere_all(response):
    df = pl.read_csv(response)
    return df.rename(
        {GEOSPHERE_RENAME_MAP}
    ).drop(["Sonnenschein", "Globalstrahlung"])

def gen_geosphere_collect(parameter, response):
    data = json.loads(response.read())
    timestamps = data.pop("timestamps")
    properties = pl.DataFrame(data["features"]).get_column("properties")
    rows = []
    for x in properties:
        rows.extend(_extract_parameter_values(x["parameters"]))
    df = pl.DataFrame(rows)
    df = df.explode("value")
    df = df.with_columns(pl.col("value").cast(pl.Float64))
    df = df.with_columns(
        pl.Series(
            Columns.DATE.value,
            pl.Series([ts for ts in timestamps for _ in range(len(parameter))])
            .cast(pl.Utf8)
            .str.to_datetime(strict=False)
            .to_list(),
        )
    )
    return df
    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: geosphere_all ===

# L1 smoke – generated
try:
    _r = gen_geosphere_all(FIX_GEOSPHERE_ALL_RESPONSE())
    print("✅ L1 smoke gen_geosphere_all: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_geosphere_all: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_geosphere_all(FIX_GEOSPHERE_ALL_RESPONSE())
    print("✅ L1 smoke before_geosphere_all: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_geosphere_all: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_geosphere_all(FIX_GEOSPHERE_ALL_RESPONSE())
    _rg = gen_geosphere_all(FIX_GEOSPHERE_ALL_RESPONSE())
    compare(_rb, _rg, "geosphere_all")
except Exception as _e:
    print(f"❌ L2 equivalence geosphere_all: setup error — {type(_e).__name__}: {_e}")

# AUDIT-55: compare schema-bearing empty CSV results.
try:
    _text = 'Sonnenschein,Globalstrahlung,id,Stationsname\n'
    _rb = before_geosphere_all(io.StringIO(_text)); _rg = gen_geosphere_all(io.StringIO(_text))
    compare(_rb, _rg, "L3 edge geosphere_all empty oracle", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge geosphere_all empty oracle: {type(_e).__name__}: {_e}")
